In [4]:
import os
import random
import zipfile
from io import BytesIO
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from huggingface_hub import HfFileSystem

HF_DATASET = "Jogitha/Animal_Image_Classification_Dataset"
ZIP_URL    = f"datasets/{HF_DATASET}/cats_dogs_snakes.zip"
WORK_DIR   = Path("animal")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK_DIR
IMG_SIZE   = (64, 64)
SEED       = 42
IMG_EXTS   = {".jpg", ".jpeg"}

random.seed(SEED)
np.random.seed(SEED)

In [5]:
# Read the zip from Hugging Face into memory (nothing saved to disk)
with HfFileSystem().open(ZIP_URL, "rb") as fh:
    zf = zipfile.ZipFile(BytesIO(fh.read()))

def load_image(path):
    """Open one image from the in-memory zip."""
    return Image.open(BytesIO(zf.read(path))).convert("RGB")

paths, labels = [], []
for name in sorted(zf.namelist()):
    if Path(name).suffix.lower() in IMG_EXTS and "__MACOSX" not in name:
        paths.append(name)
        labels.append(Path(name).parent.name.lower())   # folder name = class

classes = sorted(set(labels))
print(f"Found {len(paths)} images in {len(classes)} classes:")
for c, n in sorted(Counter(labels).items()):
    print(f"  {c:<20} {n}")

Found 3000 images in 3 classes:
  cats                 1000
  dogs                 1000
  snakes               1000


In [6]:
import shutil

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)   # start clean, so re-running never leaves old/duplicate files
ALL_DIR = OUTPUT_DIR

def downscale(src, dst):
    img = Image.open(BytesIO(zf.read(src)))  
    img = ImageOps.exif_transpose(img).convert("RGB")       # fix rotation, force 3 channels
    img = ImageOps.fit(img, IMG_SIZE, method=Image.LANCZOS)  # crop to square + anti-aliased resize
    img.save(dst)

small_paths = []                                  # paths of the 64x64 images, same order as `labels`
for i, (src, cls) in enumerate(zip(paths, labels)):
    out_dir = ALL_DIR / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    dst = out_dir / f"{cls}_{i:05d}.png"
    downscale(src, dst)
    small_paths.append(dst)

print(f"Downscaled {len(small_paths)} images to {IMG_SIZE[0]}x{IMG_SIZE[1]} -> {ALL_DIR}")

Downscaled 3000 images to 64x64 -> animal


In [7]:
train_p, temp_p, train_y, temp_y = train_test_split(
    small_paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED)

splits = {"train": (train_p, train_y), "val": (val_p, val_y), "test": (test_p, test_y)}
for name, (p_list, y_list) in splits.items():
    for src, cls in zip(p_list, y_list):
        out_dir = OUTPUT_DIR / name / cls
        out_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, out_dir / src.name)
    print(f"{name:<5}: {len(p_list):>6} images  {dict(sorted(Counter(y_list).items()))}")

train:   2100 images  {'cats': 700, 'dogs': 700, 'snakes': 700}
val  :    450 images  {'cats': 150, 'dogs': 150, 'snakes': 150}
test :    450 images  {'cats': 150, 'dogs': 150, 'snakes': 150}
